# HM3D Observation and VLM Branch Structure

This notebook asks three research questions on the real HM3D ObjectNav expert dataset.

1. **Environmental observation level:** do raw RGB observations around expert trajectories contain visible cues for branch-like decisions?
2. **Pretrained VLM representation:** do cached Prismatic VLM visual-token representations preserve branch-relevant structure beyond raw action statistics?
3. **Language involvement:** does the pretrained VLM's language readout or prompt-conditioned feature change the representation of the same real observation?

This is not a toy example. The notebook fails if the real HM3D manifests or cached payloads are missing. It uses the 6,000 materialized HM3D ObjectNav expert episodes under `/data/topovlm/habitat`.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ACTION_NAMES = {0: "STOP", 1: "MOVE_FORWARD", 2: "TURN_LEFT", 3: "TURN_RIGHT"}
TURN_ACTIONS = {2, 3}
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_ROOT = Path(os.environ.get("TOPOVLM_DATA_ROOT", "/data/topovlm/habitat"))
EPISODE_MANIFEST = DATA_ROOT / "episodes/pr2l_hm3d_objectnav/train/manifest.jsonl"
GRAPH_MANIFEST = DATA_ROOT / "graphs/pr2l_hm3d_bc/train/manifest.jsonl"

MAX_RANKING_EPISODES = int(os.environ.get("TOPOVLM_NOTEBOOK_MAX_RANKING_EPISODES", "6000"))
TARGET_EPISODES = int(os.environ.get("TOPOVLM_NOTEBOOK_TARGET_EPISODES", "80"))
MAX_FRAMES = int(os.environ.get("TOPOVLM_NOTEBOOK_MAX_FRAMES", "1200"))
MAX_VLM_NODES = int(os.environ.get("TOPOVLM_NOTEBOOK_MAX_VLM_NODES", "1500"))
RUN_LANGUAGE_CONDITIONED_VLM = os.environ.get("TOPOVLM_RUN_LANGUAGE_CONDITIONED_VLM", "0") == "1"

for required_path in (EPISODE_MANIFEST, GRAPH_MANIFEST):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print("data_root", DATA_ROOT)
print("episode_manifest", EPISODE_MANIFEST)
print("graph_manifest", GRAPH_MANIFEST)
print("run_language_conditioned_vlm", RUN_LANGUAGE_CONDITIONED_VLM)

## Load real HM3D expert episodes

The sampling target is not the first few rows, because the first manifest entries include immediate-stop episodes. We rank real episodes by trajectory length and turn pressure, then keep a diverse set across scenes and object categories.

In [ ]:
def load_jsonl(path: Path, *, limit=None) -> list[dict[str, object]]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                records.append(json.loads(line))
            if limit is not None and len(records) >= limit:
                break
    return records


def resolve_data_path(relative_path: str) -> Path:
    path = Path(relative_path)
    if path.is_absolute():
        return path
    return DATA_ROOT / path


def load_actions(record: dict[str, object]) -> np.ndarray:
    return np.load(resolve_data_path(str(record["actions_path"])), mmap_mode="r")


def load_rgb(record: dict[str, object]) -> np.ndarray:
    return np.load(resolve_data_path(str(record["rgb_path"])), mmap_mode="r")


def first_turn_label(actions: np.ndarray) -> str:
    for action in actions:
        action = int(action)
        if action == 2:
            return "first_left"
        if action == 3:
            return "first_right"
        if action == 0:
            return "stop_before_turn"
    return "no_turn"


def future_action_label(actions: np.ndarray, frame_index: int, horizon: int = 8) -> str:
    future = [int(x) for x in actions[frame_index : min(len(actions), frame_index + horizon)]]
    if not future:
        return "end"
    if 0 in future:
        return "stop_soon"
    if any(x == 2 for x in future) and any(x == 3 for x in future):
        return "mixed_turns"
    if any(x == 2 for x in future):
        return "left_soon"
    if any(x == 3 for x in future):
        return "right_soon"
    return "forward"


def is_turn_boundary(actions: np.ndarray, frame_index: int) -> bool:
    if frame_index <= 0 or frame_index >= len(actions):
        return False
    return int(actions[frame_index - 1]) not in TURN_ACTIONS and int(actions[frame_index]) in TURN_ACTIONS


def branch_candidate_score(actions: np.ndarray, frame_index: int, horizon: int = 12) -> float:
    future = [int(x) for x in actions[frame_index : min(len(actions), frame_index + horizon)]]
    if not future:
        return 0.0
    counts = Counter(future)
    total = sum(counts.values())
    entropy = 0.0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log2(p)
    turn_bonus = 0.5 if any(action in TURN_ACTIONS for action in future) else 0.0
    stop_bonus = 0.25 if 0 in future else 0.0
    return float(entropy + turn_bonus + stop_bonus)


def episode_signature(record: dict[str, object], actions: np.ndarray) -> dict[str, object]:
    return {
        "episode_id": str(record["episode_id"]),
        "scene_id": str(record["scene_id"]),
        "object_category": str(record.get("object_category") or record["goal_text"]),
        "goal_text": str(record["goal_text"]),
        "length": int(len(actions)),
        "turn_steps": int(np.isin(actions, list(TURN_ACTIONS)).sum()),
        "first_turn": first_turn_label(actions),
    }

records = load_jsonl(EPISODE_MANIFEST, limit=MAX_RANKING_EPISODES)
ranked = []
for record in records:
    actions = load_actions(record)
    signature = episode_signature(record, actions)
    score = signature["length"] + 2.0 * signature["turn_steps"]
    ranked.append((score, signature, record))
ranked.sort(key=lambda item: item[0], reverse=True)

selected_records = []
seen_scene_object = set()
for _, signature, record in ranked:
    key = (signature["scene_id"], signature["object_category"])
    if key in seen_scene_object:
        continue
    if signature["length"] < 12 or signature["turn_steps"] < 3:
        continue
    selected_records.append(record)
    seen_scene_object.add(key)
    if len(selected_records) >= TARGET_EPISODES:
        break

if len(selected_records) < min(20, TARGET_EPISODES):
    raise RuntimeError(f"Only selected {len(selected_records)} nontrivial real episodes.")

summary_rows = []
for record in selected_records[:10]:
    summary_rows.append(episode_signature(record, load_actions(record)))

print("available_records", len(records))
print("selected_nontrivial_records", len(selected_records))
print("sample_selected")
for row in summary_rows:
    print(json.dumps(row, sort_keys=True))

## 1. Environmental observation level

This section does not use learned VLM latents. It embeds raw RGB observations through low-level image descriptors and asks whether branch-related trajectory labels organize the observation space.

The plot is evidence only for **visible observation cues**. It is not yet proof that the dataset has true topology, because visual similarity can reflect object category, scene identity, lighting, or camera pose.

In [ ]:
def select_frame_indices(actions: np.ndarray, *, max_per_episode: int = 16) -> list[int]:
    candidates = set()
    if len(actions) == 0:
        return []
    stride = max(1, len(actions) // max(1, max_per_episode // 2))
    candidates.update(range(0, len(actions), stride))
    for idx in range(1, len(actions)):
        if is_turn_boundary(actions, idx):
            for offset in (-2, -1, 0, 1, 2):
                j = idx + offset
                if 0 <= j < len(actions):
                    candidates.add(j)
    ranked_indices = sorted(candidates, key=lambda i: branch_candidate_score(actions, i), reverse=True)
    keep = sorted(ranked_indices[:max_per_episode])
    return keep


def rgb_descriptor(frame: np.ndarray) -> np.ndarray:
    image = Image.fromarray(frame.astype("uint8")).resize((32, 24), Image.BILINEAR)
    arr = np.asarray(image, dtype=np.float32) / 255.0
    color_mean = arr.reshape(-1, 3).mean(axis=0)
    color_std = arr.reshape(-1, 3).std(axis=0)
    gray = arr.mean(axis=2)
    dx = np.diff(gray, axis=1, prepend=gray[:, :1])
    dy = np.diff(gray, axis=0, prepend=gray[:1, :])
    edge = np.stack([np.abs(dx), np.abs(dy)], axis=2)
    return np.concatenate([arr.reshape(-1), edge.reshape(-1), color_mean, color_std]).astype(np.float32)

obs_features = []
obs_rows = []
for record in selected_records:
    actions = np.asarray(load_actions(record))
    rgb = load_rgb(record)
    for frame_index in select_frame_indices(actions, max_per_episode=16):
        if len(obs_rows) >= MAX_FRAMES:
            break
        frame = np.asarray(rgb[frame_index])
        obs_features.append(rgb_descriptor(frame))
        obs_rows.append({
            "episode_id": record["episode_id"],
            "scene_id": record["scene_id"],
            "object_category": record.get("object_category") or record["goal_text"],
            "goal_text": record["goal_text"],
            "frame_index": int(frame_index),
            "action": ACTION_NAMES[int(actions[frame_index])],
            "future_action": future_action_label(actions, frame_index),
            "branch_score": branch_candidate_score(actions, frame_index),
            "turn_boundary": is_turn_boundary(actions, frame_index),
        })
    if len(obs_rows) >= MAX_FRAMES:
        break

obs_features = np.asarray(obs_features, dtype=np.float32)
if obs_features.shape[0] < 100:
    raise RuntimeError(f"Need at least 100 real frames for observation embedding, got {obs_features.shape[0]}")

scaled_obs = StandardScaler().fit_transform(obs_features)
obs_pca50 = PCA(n_components=min(50, scaled_obs.shape[0] - 1), random_state=RANDOM_SEED).fit_transform(scaled_obs)
obs_xy = TSNE(n_components=2, perplexity=min(30, max(5, obs_pca50.shape[0] // 20)), init="pca", learning_rate="auto", random_state=RANDOM_SEED).fit_transform(obs_pca50)

print("observation_frames", len(obs_rows), "feature_dim", obs_features.shape[1])
print("future_action_counts", Counter(row["future_action"] for row in obs_rows))

In [ ]:
def encode_labels(values: list[str]) -> tuple[np.ndarray, list[str]]:
    unique = sorted(set(values))
    mapping = {value: idx for idx, value in enumerate(unique)}
    return np.asarray([mapping[value] for value in values]), unique


def scatter_by_label(xy: np.ndarray, labels: list[str], title: str, ax: plt.Axes):
    codes, unique = encode_labels(labels)
    scatter = ax.scatter(xy[:, 0], xy[:, 1], c=codes, s=12, alpha=0.78, cmap="tab10")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    handles = []
    for value, code in zip(unique, range(len(unique))):
        handles.append(plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=scatter.cmap(scatter.norm(code)), markersize=6, label=value))
    ax.legend(handles=handles, fontsize=7, loc="best", frameon=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
scatter_by_label(obs_xy, [row["future_action"] for row in obs_rows], "Raw observation embedding: future action", axes[0])
scatter_by_label(obs_xy, [row["action"] for row in obs_rows], "Raw observation embedding: current action", axes[1])
branch_scores = np.asarray([row["branch_score"] for row in obs_rows], dtype=float)
plot = axes[2].scatter(obs_xy[:, 0], obs_xy[:, 1], c=branch_scores, s=12, alpha=0.78, cmap="viridis")
axes[2].set_title("Raw observation embedding: branch-candidate score")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(plot, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

future_codes, future_names = encode_labels([row["future_action"] for row in obs_rows])
if len(set(future_codes)) > 1:
    score = silhouette_score(obs_xy, future_codes)
    print("2D silhouette_by_future_action", round(float(score), 4), "labels", future_names)

In [ ]:
def show_branch_candidate_frames(rows: list[dict[str, object]], *, n: int = 24):
    top_rows = sorted(rows, key=lambda row: row["branch_score"], reverse=True)[:n]
    columns = 6
    rows_count = math.ceil(len(top_rows) / columns)
    fig, axes = plt.subplots(rows_count, columns, figsize=(15, 2.5 * rows_count))
    axes = np.asarray(axes).reshape(-1)
    record_by_id = {record["episode_id"]: record for record in selected_records}
    for ax, row in zip(axes, top_rows):
        record = record_by_id[row["episode_id"]]
        rgb = load_rgb(record)
        frame = np.asarray(rgb[row["frame_index"]])
        ax.imshow(frame)
        ax.set_title(f"{row['future_action']} | {row['object_category']}\nscore={row['branch_score']:.2f} frame={row['frame_index']}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(top_rows):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_branch_candidate_frames(obs_rows, n=24)

## 2. Cached pretrained VLM representation

The repo already contains Prismatic visual-token cache for the same real HM3D episodes. This section pools cached token features and asks whether the pretrained representation organizes observations by branch-related trajectory labels.

This is stronger than raw RGB descriptors, but still not a final topology claim. The cached representation can encode object and room semantics, generated-text context, and other visual regularities that are not necessarily future reachability.

In [ ]:
graph_records = load_jsonl(GRAPH_MANIFEST)
graph_by_episode = {record["episode_id"]: record for record in graph_records}

vlm_features = []
vlm_rows = []
for record in selected_records:
    graph_record = graph_by_episode.get(record["episode_id"])
    if graph_record is None:
        continue
    actions = np.asarray(load_actions(record))
    embedding_path = resolve_data_path(str(graph_record["embedding_path"]))
    embeddings = np.load(embedding_path, mmap_mode="r")
    if embeddings.ndim == 3:
        pooled = np.asarray(embeddings).mean(axis=1)
    elif embeddings.ndim == 2:
        pooled = np.asarray(embeddings)
    else:
        raise ValueError((embedding_path, embeddings.shape))
    for node_index in range(pooled.shape[0]):
        if len(vlm_rows) >= MAX_VLM_NODES:
            break
        action_index = min(node_index, len(actions) - 1)
        vlm_features.append(pooled[node_index].astype(np.float32))
        vlm_rows.append({
            "episode_id": record["episode_id"],
            "scene_id": record["scene_id"],
            "object_category": record.get("object_category") or record["goal_text"],
            "goal_text": record["goal_text"],
            "node_index": int(node_index),
            "action_index": int(action_index),
            "action": ACTION_NAMES[int(actions[action_index])],
            "future_action": future_action_label(actions, action_index),
            "branch_score": branch_candidate_score(actions, action_index),
            "representation_id": graph_record.get("representation_id"),
        })
    if len(vlm_rows) >= MAX_VLM_NODES:
        break

vlm_features = np.asarray(vlm_features, dtype=np.float32)
if vlm_features.shape[0] < 100:
    raise RuntimeError(f"Need at least 100 cached VLM vectors, got {vlm_features.shape[0]}")

scaled_vlm = StandardScaler().fit_transform(vlm_features)
vlm_pca50 = PCA(n_components=min(50, scaled_vlm.shape[0] - 1), random_state=RANDOM_SEED).fit_transform(scaled_vlm)
vlm_xy = TSNE(n_components=2, perplexity=min(30, max(5, vlm_pca50.shape[0] // 20)), init="pca", learning_rate="auto", random_state=RANDOM_SEED).fit_transform(vlm_pca50)

print("vlm_vectors", len(vlm_rows), "feature_dim", vlm_features.shape[1])
print("representation", vlm_rows[0]["representation_id"])
print("future_action_counts", Counter(row["future_action"] for row in vlm_rows))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
scatter_by_label(vlm_xy, [row["future_action"] for row in vlm_rows], "Prismatic cached latent: future action", axes[0])
scatter_by_label(vlm_xy, [row["action"] for row in vlm_rows], "Prismatic cached latent: current action", axes[1])
plot = axes[2].scatter(vlm_xy[:, 0], vlm_xy[:, 1], c=[row["branch_score"] for row in vlm_rows], s=12, alpha=0.78, cmap="viridis")
axes[2].set_title("Prismatic cached latent: branch-candidate score")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(plot, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

future_codes, future_names = encode_labels([row["future_action"] for row in vlm_rows])
if len(set(future_codes)) > 1:
    score = silhouette_score(vlm_xy, future_codes)
    print("2D silhouette_by_future_action", round(float(score), 4), "labels", future_names)

In [ ]:
def load_metadata(record: dict[str, object]):
    metadata_path = record.get("metadata_path")
    if metadata_path is None:
        return None
    path = resolve_data_path(str(metadata_path))
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

metadata_rows = []
for record in selected_records[: min(len(selected_records), 80)]:
    graph_record = graph_by_episode.get(record["episode_id"])
    if graph_record is None:
        continue
    metadata = load_metadata(graph_record)
    if metadata is None:
        continue
    generated_texts = metadata.get("generated_texts") or []
    actions = np.asarray(load_actions(record))
    metadata_rows.append({
        "episode_id": record["episode_id"],
        "object_category": record.get("object_category") or record["goal_text"],
        "first_turn": first_turn_label(actions),
        "turn_steps": int(np.isin(actions, list(TURN_ACTIONS)).sum()),
        "generated_text": " | ".join(str(text) for text in generated_texts[:3]),
        "include_generated_text": metadata.get("include_generated_text"),
        "prompt_template": metadata.get("prompt_template"),
    })

print("metadata_rows", len(metadata_rows))
print("prompt_template", metadata_rows[0]["prompt_template"] if metadata_rows else None)
print("examples")
for row in metadata_rows[:12]:
    print(json.dumps(row, sort_keys=True))

## 3. Language-conditioned VLM check

The broad cached representation above is useful, but it does not by itself prove language-conditioned branch sensitivity. This section uses the same real HM3D frames and runs a small Prismatic prompt sweep when `TOPOVLM_RUN_LANGUAGE_CONDITIONED_VLM=1` is set in a GPU runtime.

The question is: for the same observation, does the VLM representation move when the language asks for different future branches or goals?

In [ ]:
def selected_language_probe_frames(rows: list[dict[str, object]], *, n: int = 6) -> list[dict[str, object]]:
    top_rows = sorted(rows, key=lambda row: row["branch_score"], reverse=True)
    chosen = []
    seen_episode = set()
    for row in top_rows:
        if row["episode_id"] in seen_episode:
            continue
        chosen.append(row)
        seen_episode.add(row["episode_id"])
        if len(chosen) >= n:
            break
    return chosen

language_probe_rows = selected_language_probe_frames(obs_rows, n=6)
for row in language_probe_rows:
    print(json.dumps({k: row[k] for k in ["episode_id", "object_category", "frame_index", "future_action", "branch_score"]}, sort_keys=True))

In [ ]:
if not RUN_LANGUAGE_CONDITIONED_VLM:
    print("Skipping Prismatic forward pass. Set TOPOVLM_RUN_LANGUAGE_CONDITIONED_VLM=1 in a GPU session to run this real-data prompt sweep.")
else:
    import torch
    from configs import build_config_from_exp
    from encoders.prismatic import PrismaticEncoder

    if not torch.cuda.is_available():
        raise RuntimeError("Language-conditioned VLM analysis requires CUDA for the 7B Prismatic model.")

    cfg = build_config_from_exp("habitat/pr2l_hm3d_bc")
    cfg.model.vlm.weights_path = "/data/topovlm/vlm_weights/prismatic/prism-dinosiglip+7b"
    cfg.model.vlm.prompt_template = "{goal_text}"
    cfg.model.vlm.include_generated_text = False
    cfg.model.vlm.representation = "last_token"
    cfg.model.vlm.output_dim = 4096
    encoder = PrismaticEncoder(cfg.model.vlm)

    prompt_templates = [
        "What future route is likely from here if the goal is {goal}?",
        "Does this observation support turning left toward a different future state?",
        "Does this observation support turning right toward a different future state?",
        "Does this observation look like a straight corridor with one likely continuation?",
    ]

    conditioned_features = []
    conditioned_rows = []
    record_by_id = {record["episode_id"]: record for record in selected_records}
    for probe in language_probe_rows:
        record = record_by_id[probe["episode_id"]]
        rgb = load_rgb(record)
        image = Image.fromarray(np.asarray(rgb[probe["frame_index"]]).astype("uint8"))
        for prompt_template in prompt_templates:
            prompt = prompt_template.format(goal=probe["object_category"])
            feature = encoder.encode_image_goal(image, prompt)
            conditioned_features.append(feature.astype(np.float32))
            conditioned_rows.append({
                "episode_id": probe["episode_id"],
                "object_category": probe["object_category"],
                "frame_index": probe["frame_index"],
                "future_action": probe["future_action"],
                "branch_score": probe["branch_score"],
                "prompt": prompt,
            })

    conditioned_features = np.asarray(conditioned_features, dtype=np.float32)
    conditioned_xy = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(StandardScaler().fit_transform(conditioned_features))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    scatter_by_label(conditioned_xy, [row["prompt"] for row in conditioned_rows], "Same real frames: prompt-conditioned VLM features", axes[0])
    scatter_by_label(conditioned_xy, [row["episode_id"] for row in conditioned_rows], "Same features grouped by observation", axes[1])
    plt.tight_layout()
    plt.show()

    normalized = conditioned_features / np.linalg.norm(conditioned_features, axis=1, keepdims=True)
    cosine = normalized @ normalized.T
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title("Prompt-conditioned feature cosine similarity")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## How to read the result

A useful topology-like result should satisfy more than visual separation in 2D.

- Raw observation embedding should show branch-candidate frames that are visually meaningful, not just scene or object clusters.
- Cached VLM latent embedding should preserve future-action or branch-score organization beyond object category.
- Language-conditioned features should move for the same observation when prompts ask for different future branches.
- A paper-facing claim needs agreement with geometry, future-state prediction, or held-out scenes; the embedding alone is a discovery tool, not proof.

In [ ]:
print("Notebook completed setup and visualization cells.")
print("Real dataset root:", DATA_ROOT)
print("Observation frames analyzed:", len(obs_rows))
print("Cached VLM vectors analyzed:", len(vlm_rows))
print("Language-conditioned VLM forward enabled:", RUN_LANGUAGE_CONDITIONED_VLM)